# 算法定价与合谋 / Algorithmic pricing and collusion
Calvano et al. (AER 2020), DOI 10.1257/aer.20190623.
部分课堂复现，非完整论文复现。 / Partial classroom reproduction, not a full-paper replication.
此笔记本不调用付费API。 / No paid API calls.


In [ ]:
%pip -q install numpy matplotlib


In [ ]:
"""Independent Python implementation of Calvano et al. (AER 2020), duopoly k=1.

No API calls or business data. The authors' original code has not been audited.
"""
from dataclasses import dataclass, asdict
import math
import time
import numpy as np

@dataclass(frozen=True)
class Config:
    cost: float = 1.0
    quality: float = 2.0
    outside: float = 0.0
    mu: float = 0.25
    discount: float = 0.95
    learning_rate: float = 0.15
    beta: float = 4e-6
    actions: int = 15
    extension: float = 0.1
    stable_periods: int = 100_000

def shares(p1, p2, c):
    e1 = np.exp((c.quality-np.asarray(p1))/c.mu)
    e2 = np.exp((c.quality-np.asarray(p2))/c.mu)
    den = e1+e2+math.exp(c.outside/c.mu)
    return e1/den, e2/den

def bisect(fn, lo, hi):
    if fn(lo)*fn(hi) >= 0: raise ValueError('Root not bracketed')
    for _ in range(100):
        mid=(lo+hi)/2
        if fn(lo)*fn(mid) <= 0: hi=mid
        else: lo=mid
    return (lo+hi)/2

def environment(c):
    # Symmetric continuous-price Nash and joint-profit FOCs.
    nash=bisect(lambda p: p-c.cost-c.mu/(1-shares(p,p,c)[0]),c.cost,5.)
    monopoly=bisect(lambda p: p-c.cost-c.mu/(1-2*shares(p,p,c)[0]),c.cost,5.)
    grid=np.linspace(nash-c.extension*(monopoly-nash),monopoly+c.extension*(monopoly-nash),c.actions)
    a,b=np.meshgrid(grid,grid,indexing='ij')
    q1,q2=shares(a,b,c)
    profit=np.stack([(a-c.cost)*q1,(b-c.cost)*q2],axis=-1)
    pn=float((nash-c.cost)*shares(nash,nash,c)[0])
    pm=float((monopoly-c.cost)*shares(monopoly,monopoly,c)[0])
    return grid,profit,dict(nash_price=nash,monopoly_price=monopoly,nash_profit=pn,monopoly_profit=pm)

def frozen_cycle(policy, state, m):
    seen={}; sequence=[]
    while state not in seen:
        seen[state]=len(sequence)
        a=int(policy[0,state]); b=int(policy[1,state])
        sequence.append((state,a,b)); state=a*m+b
    return sequence[seen[state]:]

def deviation_path(policy, state, grid, profit, periods=30):
    """Freeze learning/exploration; one-period myopic best deviation by firm 1.

    Classroom adaptation, not a claimed exact replication of Figure 6's protocol.
    """
    out=[];m=len(grid)
    for t in range(periods):
        a=int(policy[0,state]);b=int(policy[1,state])
        if t==5: a=int(np.argmax(profit[:,b,0]))
        out.append(dict(period=t,firm1_price=float(grid[a]),firm2_price=float(grid[b]),forced_deviation=t==5))
        state=a*m+b
    return out

def run(c,seed,max_steps=2_000_000,block=10_000,output=None):
    grid,profit,reference=environment(c);m=c.actions;n=m*m
    # Initial Q is discounted profit against a uniformly random rival, all states.
    initial=np.stack([profit[:,:,0].mean(axis=1),profit[:,:,1].mean(axis=0)])/(1-c.discount)
    q=np.repeat(initial[:,None,:],n,axis=1)
    policy=q.argmax(axis=2); value=q.max(axis=2)
    rng=np.random.default_rng(seed);state=int(rng.integers(n));stable=0
    history=[];total_prices=np.zeros(2);total_profits=np.zeros(2);count=0;beg=time.perf_counter()
    for t in range(max_steps):
        if t%block==0: randoms=rng.random((min(block,max_steps-t),4))
        r=randoms[t%block];eps=math.exp(-c.beta*t)
        a=int(r[2]*m) if r[0]<eps else int(policy[0,state])
        b=int(r[3]*m) if r[1]<eps else int(policy[1,state])
        next_state=a*m+b
        # Both targets are computed before either Q-table changes.
        target0=profit[a,b,0]+c.discount*value[0,next_state]
        target1=profit[a,b,1]+c.discount*value[1,next_state]
        changed=False
        for i,action,target in [(0,a,target0),(1,b,target1)]:
            q[i,state,action]+=c.learning_rate*(target-q[i,state,action])
            # argmax selects the lowest price on ties, as specified in the article.
            new=int(q[i,state].argmax());changed|=new!=policy[i,state]
            policy[i,state]=new;value[i,state]=q[i,state,new]
        stable=0 if changed else stable+1
        state=next_state;total_prices+=grid[[a,b]];total_profits+=profit[a,b];count+=1
        if count==block or stable>=c.stable_periods or t==max_steps-1:
            history.append(dict(step=t+1,epsilon=eps,firm1_price=float(total_prices[0]/count),firm2_price=float(total_prices[1]/count),profit_per_firm=float(total_profits.mean()/count)))
            total_prices[:]=0;total_profits[:]=0;count=0
        if stable>=c.stable_periods: break
    cycle=frozen_cycle(policy,state,m)
    avgprofit=float(np.mean([profit[a,b].mean() for _,a,b in cycle]))
    result=dict(config=asdict(c),seed=seed,steps=t+1,max_steps=max_steps,converged=stable>=c.stable_periods,stable_periods_observed=stable,seconds=time.perf_counter()-beg,reference=reference,cycle_length=len(cycle),frozen_mean_price=float(np.mean([grid[[a,b]] for _,a,b in cycle])),frozen_profit_per_firm=avgprofit,profit_gain=(avgprofit-reference['nash_profit'])/(reference['monopoly_profit']-reference['nash_profit']),history=history,deviation=deviation_path(policy,cycle[0][0],grid,profit))
    if output is not None: np.savez_compressed(output,q=q,policy=policy,prices=grid,state=state)
    return result


## 运行 / Run
达到步数上限不代表收敛。 / Reaching the step cap does not establish convergence.
不同参数请使用新的输出目录。 / Use a new output directory for different parameters.


In [ ]:
# 三次训练可能耗时数分钟。 / Three runs may take several minutes.
from pathlib import Path
import json
out=Path("results"); out.mkdir(exist_ok=True)
runs=[]
for seed in [0,1,2]:
    dest=out/f"seed_{seed}.json"
    if dest.exists():
        result=json.loads(dest.read_text())
        assert result['seed']==seed and result['config']==asdict(Config())
        assert result['max_steps']==2_000_000
    else:
        result=run(Config(),seed,2_000_000,output=out/f"seed_{seed}.npz")
        dest.write_text(json.dumps(result,indent=2))
    runs.append(result)
    print(seed,result['converged'],result['frozen_mean_price'],result['profit_gain'])


In [ ]:
import matplotlib.pyplot as plt
fig, ax=plt.subplots(figsize=(10,5))
for r in runs:
    h=r['history']
    ax.plot([x['step'] for x in h],
            [(x['firm1_price']+x['firm2_price'])/2 for x in h],
            label=f"seed {r['seed']}")
ref=runs[0]['reference']
ax.axhline(ref['nash_price'],color='black',ls='--',label='Nash')
ax.axhline(ref['monopoly_price'],color='gray',ls='--',label='Monopoly')
ax.set(xlabel='Training period',ylabel='Mean price per 10,000-period block',
       title='Limited Q-learning reproduction')
ax.legend(); fig.tight_layout()
fig.savefig('price_evolution.png',dpi=200); plt.show()


## 解释 / Interpretation
高于竞争价格不足以证明合谋。还需策略稳定性、偏离惩罚及最佳反应检验。
Prices above Nash alone do not establish collusion. Policy stability, punishment and best-response tests are additional requirements.
保存JSON、NPZ和图片后再关闭Colab。 / Save JSON, NPZ, and figures before closing Colab.


## 第二阶段 / Stage 2
最佳反应与一期降价验证。运行前完成三个训练。 / Best-response and one-period price-cut diagnostics. Complete the three training runs first.


In [ ]:
"""Exact finite-state best-response diagnostics for saved frozen policies.

Q-learning is disabled. Best responses optimize discounted infinite-horizon
profit against the opponent's fixed memory-one strategy, on the same price grid.
The deviation experiment is an explicitly specified classroom adaptation.
"""
from pathlib import Path
import json,csv
import numpy as np

ROOT=Path.cwd()

def mdp(policy, player, profit, m):
    other=policy[1-player]
    actions=np.arange(m)[None,:]
    if player==0:
        nxt=actions*m+other[:,None]
        rewards=profit[actions,other[:,None],0]
    else:
        nxt=other[:,None]*m+actions
        rewards=profit[other[:,None],actions,1]
    return nxt,rewards

def values(policy, player, profit, discount, tol=1e-11):
    m=profit.shape[0];states=np.arange(m*m);nxt,rewards=mdp(policy,player,profit,m)
    chosen=policy[player];pi_next=nxt[states,chosen];pi_reward=rewards[states,chosen]
    v=np.zeros(m*m)
    for _ in range(10000):
        new=pi_reward+discount*v[pi_next]
        if np.max(np.abs(new-v))<tol: v=new;break
        v=new
    else: raise RuntimeError('Policy evaluation failed to converge')
    optimum=np.zeros(m*m)
    for _ in range(10000):
        new=np.max(rewards+discount*optimum[nxt],axis=1)
        if np.max(np.abs(new-optimum))<tol: optimum=new;break
        optimum=new
    else: raise RuntimeError('Best-response value iteration failed to converge')
    residual=float(np.max(np.abs(np.max(rewards+discount*optimum[nxt],axis=1)-optimum)))
    q=rewards+discount*v[nxt]
    own_residual=float(np.max(np.abs(pi_reward+discount*v[pi_next]-v)))
    assert np.all(optimum>=v-1e-8)
    return v,optimum,q,residual,own_residual

def paths(policy, start, player, grid, profit, horizon=100):
    """At t=5 force the highest-stage-profit strictly lower price for one period.

    Both players otherwise follow their frozen learned policies. Simultaneous
    actions imply the nondeviator cannot react until t=6.
    """
    m=len(grid);baseline=start;deviated=start;rows=[];cycle_states={s for s,_,_ in frozen_cycle(policy,start,m)}
    return_time=None;forced=None
    for t in range(horizon):
        normal=[int(policy[i,baseline]) for i in [0,1]]
        actual=[int(policy[i,deviated]) for i in [0,1]]
        if t==5:
            current=actual[player]
            if current==0: raise ValueError('No strictly lower price on grid')
            candidate=profit[:current,actual[1],0] if player==0 else profit[actual[0],:current,1]
            actual[player]=int(np.argmax(candidate));forced=actual[player]
        record=dict(period=t,baseline_firm1=float(grid[normal[0]]),baseline_firm2=float(grid[normal[1]]),deviation_firm1=float(grid[actual[0]]),deviation_firm2=float(grid[actual[1]]),forced=t==5,baseline_state=baseline,deviation_state=deviated)
        rows.append(record)
        if t>5 and deviated in cycle_states and return_time is None: return_time=t-5
        baseline=normal[0]*m+normal[1];deviated=actual[0]*m+actual[1]
    before=rows[5][f'baseline_firm{player+1}'];after=rows[5][f'deviation_firm{player+1}']
    other=1-player
    reaction=rows[6][f'deviation_firm{other+1}']-rows[6][f'baseline_firm{other+1}']
    return dict(deviator=player+1,forced_before=before,forced_after=after,
                opponent_next_period_price_change=reaction,return_to_original_cycle_after_periods=return_time,horizon=horizon,rows=rows)

def main():
    all_results=[]
    for f in sorted((ROOT/'results').glob('seed_*.json')):
        r=json.loads(f.read_text(encoding='utf-8'));seed=r['seed'];c=Config(**r['config'])
        saved=np.load(ROOT/'results'/f'seed_{seed}.npz',allow_pickle=False)
        policy=saved['policy'];state=int(saved['state']);grid,profit,ref=environment(c)
        assert np.array_equal(grid,saved['prices'])
        assert np.array_equal(policy,saved['q'].argmax(axis=2))
        cycle=frozen_cycle(policy,state,c.actions);states=np.array([s for s,_,_ in cycle])
        diagnostics=[]
        for i in [0,1]:
            v,opt,q,residual,own_residual=values(policy,i,profit,c.discount)
            gain=opt-v;relative=100*gain/opt
            diagnostics.append(dict(player=i+1,max_discounted_gain_on_cycle=float(gain[states].max()),max_percent_value_loss_on_cycle=float(relative[states].max()),max_percent_value_loss_all_states=float(relative.max()),on_cycle_best_response=bool(gain[states].max()<1e-7),all_states_best_response=bool(gain.max()<1e-7),optimal_bellman_residual=residual,policy_bellman_residual=own_residual,best_response_value_error_bound=residual/(1-c.discount)))
        all_results.append(dict(seed=seed,training_converged=r['converged'],cycle_length=len(cycle),joint_on_cycle_best_response=all(x['on_cycle_best_response'] for x in diagnostics),joint_all_states_best_response=all(x['all_states_best_response'] for x in diagnostics),players=diagnostics,deviations=[paths(policy,cycle[0][0],i,grid,profit) for i in [0,1]]))
    (ROOT/'results/strategy_diagnostics.json').write_text(json.dumps(all_results,indent=2),encoding='utf-8')
    for r in all_results:
        print(json.dumps({k:v for k,v in r.items() if k!='deviations'},indent=2))
        print('Deviations:',[{k:v for k,v in x.items() if k!='rows'} for x in r['deviations']])
    with (ROOT/'results/strategy_summary.csv').open('w',newline='',encoding='utf-8') as f:
        fields=['seed','training_converged','player','max_percent_value_loss_on_cycle','max_percent_value_loss_all_states','on_cycle_best_response','all_states_best_response']
        w=csv.DictWriter(f,fieldnames=fields);w.writeheader()
        for r in all_results:
            for p in r['players']:w.writerow({k:dict(seed=r['seed'],training_converged=r['training_converged'],**p)[k] for k in fields})

if __name__=='__main__':main()


In [ ]:
"""Create publication-exportable diagnostic figures from saved results only."""
from pathlib import Path
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties

ROOT=Path.cwd()
cn=Path('C:/Windows/Fonts/msyh.ttc')
if cn.exists():
    import matplotlib.font_manager as fm
    fm.fontManager.addfont(str(cn))
    plt.rcParams['font.family']=FontProperties(fname=str(cn)).get_name()
plt.rcParams['axes.unicode_minus']=False
out=ROOT/'figures';out.mkdir(exist_ok=True)
runs=[json.loads(p.read_text(encoding='utf-8')) for p in sorted((ROOT/'results').glob('seed_*.json'))]
diags=json.loads((ROOT/'results/strategy_diagnostics.json').read_text(encoding='utf-8'))
fig,ax=plt.subplots(figsize=(10,5.2))
for r in runs:
    h=r['history'];label=f"Seed {r['seed']} — "+('已稳定 / stable' if r['converged'] else '未收敛 / capped')
    ax.plot([x['step'] for x in h],[(x['firm1_price']+x['firm2_price'])/2 for x in h],label=label)
ref=runs[0]['reference']
ax.axhline(ref['nash_price'],color='black',ls='--',label='竞争 / Nash')
ax.axhline(ref['monopoly_price'],color='gray',ls=':',label='垄断 / Monopoly')
ax.set(xlabel='训练期数 / Training period',ylabel='区间平均价格 / Block mean price',title='价格学习过程 / Price learning dynamics')
ax.grid(alpha=.2);ax.legend(fontsize=8);fig.tight_layout();fig.savefig(out/'price_evolution.png',dpi=200);plt.close(fig)
fig,axs=plt.subplots(2,2,figsize=(12,8),sharex=True,sharey=True)
for row,seed in enumerate([1,2]):
    r=next(x for x in diags if x['seed']==seed)
    for col,d in enumerate(r['deviations']):
        ax=axs[row,col];rows=d['rows'][:25];x=[v['period']-5 for v in rows]
        for player,color in [(1,'#2166ac'),(2,'#b35806')]:
            ax.plot(x,[v[f'deviation_firm{player}'] for v in rows],marker='o',ms=3,label=f'企业 / Firm {player}',color=color)
            ax.plot(x,[v[f'baseline_firm{player}'] for v in rows],ls=':',color=color,alpha=.6)
        ax.axvline(0,color='gray',ls='--',lw=1)
        ax.set_title(f"Seed {seed} · 降价企业 / Deviator {d['deviator']}")
        ax.grid(alpha=.2);ax.set_xlabel('相对降价期 / Period relative to deviation');ax.set_ylabel('价格 / Price')
axs[0,0].legend(fontsize=8)
fig.suptitle('冻结策略下的一期降价测试 / One-period price-cut test with frozen policies')
fig.tight_layout();fig.savefig(out/'deviation_responses.png',dpi=200);plt.close(fig)
fig,ax=plt.subplots(figsize=(9,5))
for offset,player,color in [(-.18,1,'#2166ac'),(.18,2,'#b35806')]:
    vals=[r['players'][player-1]['max_percent_value_loss_on_cycle'] for r in diags]
    bars=ax.bar([r['seed']+offset for r in diags],vals,width=.34,color=color,label=f'企业 / Firm {player}')
    ax.bar_label(bars,fmt='%.2f',padding=3)
ax.set_xticks([0,1,2],['Seed 0\n未收敛 / capped','Seed 1','Seed 2'])
ax.set(ylabel='最优价值损失 / Optimal-value loss (%)',title='稳定不等于最佳反应 / Stable policy does not imply best response',ylim=(0,22))
ax.legend();ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True);fig.tight_layout();fig.savefig(out/'best_response_gaps.png',dpi=200);plt.close(fig)
print('Three diagnostic figures saved.')
